# Sentinel-2 parcel zonal statistics

Configure and run `SatelliteZonalStats`. openEO creates one cloud-masked, indexed monthly NetCDF cube per spatial batch. IQR cleaning, null filling, and parcel statistics run locally, producing one ML-ready row per parcel and month.

In [1]:
import os

from pathlib import Path

import geopandas as gpd

from satellite_extract_data.zonal_stats import SatelliteZonalStats


## Inputs

Processing order:

`Sentinel-2 L2A ? acquisition cloud mask ? acquisition pixel indices ? monthly pixel median ? monthly per-variable cube IQR outlier removal ? local null filling ? parcel statistics ? compact table`

The bidirectional temporal fill is appropriate for retrospective feature preparation. Do not use future months to fill earlier months in a causal forecasting model.

In [2]:
WORK_PATH = Path("C:/work_dir")
PARCELS_PATH = WORK_PATH / "parcels_axios.gpkg"

START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
OUTPUT_DIR = WORK_PATH / "sentinel2_monthly_parcel_statistics_local4"
WORKING_EPSG = 2100

CALCULATE_INDICES = True
INDICES = ["NDVI", "NDWI", "NDMI", "EVI", "SAVI", "NDRE", "CI"]
BANDS = ["B02", "B03", "B04", "B05", "B08", "B11", "B12"]
# Statistics and cleaning settings are local. Changing them reuses the
# cached monthly NetCDF without submitting another openEO job.
# Mean is mandatory; percentiles enable derived IQR and robust skewness.
SPATIAL_STATISTICS = [
    "mean",
    "median",
    "sd",
    "min",
    "max",
    "count",
    "p10",
    "p25",
    "p75",
    "p90",
]

# Reuses the legacy EO order: IQR outliers become null, then nulls are filled.
REMOVE_OUTLIERS = True
IQR_QUANTILES = (0.1, 0.9)
IQR_MULTIPLIER = 1.5
FILL_NULLS = True

# Free tier limations
# Synchronous requests are limited to 2 concurrent requests
# Batch jobs are limited to 2 concurrent jobs
BATCH_WORKERS = 2

parcels = gpd.read_file(PARCELS_PATH)

## Run extraction

In [4]:
extractor = SatelliteZonalStats(
    parcels=parcels,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    working_epsg=WORKING_EPSG,
    calculate_indices=CALCULATE_INDICES,
    indices=INDICES,
    spatial_statistics=SPATIAL_STATISTICS,
    remove_outliers=REMOVE_OUTLIERS,
    iqr_quantiles=IQR_QUANTILES,
    iqr_multiplier=IQR_MULTIPLIER,
    fill_nulls=FILL_NULLS,
    
    batch_workers=BATCH_WORKERS,
)
extractor.BANDS = BANDS
extractor.PARCEL_ID_FIELD = "parcel_id"
results = extractor.run()
results.head()

2026-07-26 16:47:37,524 | INFO | satellite_extract_data.zonal_stats.1415013322256 | Prepared 37 valid parcels.
2026-07-26 16:47:37,525 | INFO | satellite_extract_data.zonal_stats.1415013322256 | Configured indices=['NDVI', 'NDWI', 'NDMI', 'EVI', 'SAVI', 'NDRE', 'CI'], parcel statistics=['mean', 'median', 'sd', 'min', 'max', 'count', 'p10', 'p25', 'p75', 'p90'], remove_outliers=True, iqr_quantiles=(0.1, 0.9), iqr_multiplier=1.5, and fill_nulls=True, temporal_period=1M, and temporal_reducer=median, and batch_workers=2.
2026-07-26 16:47:37,527 | INFO | satellite_extract_data.zonal_stats.1415013322256 | Monthly cube cache: C:\work_dir\sentinel2_monthly_parcel_statistics_local4\monthly_cubes\f699e3e0bf4bb72d.
2026-07-26 16:47:37,535 | INFO | satellite_extract_data.zonal_stats.1415013322256 | Using existing batch 1 monthly cube C:\work_dir\sentinel2_monthly_parcel_statistics_local4\monthly_cubes\f699e3e0bf4bb72d\batch_00001_monthly.nc.
2026-07-26 16:47:37,535 | INFO | satellite_extract_data.

,parcel_id,period_start,period_end,B02_mean,B02_median,B02_sd,B02_min,B02_max,B02_count,B02_p10,...,NDRE_p90_p10_spread,NDRE_valid_pixel_fraction_approx,CI_range,CI_variance,CI_coefficient_of_variation,CI_iqr,CI_bowley_skewness,CI_p90_p10_spread,CI_valid_pixel_fraction_approx,geometry
0,1,2025-01-01,2025-02-01,0.055427,0.05235,0.008058,0.04760,0.0861,149,0.05000,...,0.090870,0.999509,0.651834,0.010655,0.570677,0.066437,-0.095921,0.216638,0.999509,"MULTIPOLYGON (((22.76878 40.60366, 22.76921 40..."
1,1,2025-02-01,2025-03-01,0.080481,0.07955,0.004149,0.07355,0.0986,149,0.07669,...,0.038976,0.999509,0.400034,0.002614,0.289074,0.040734,-0.025815,0.092258,0.999509,"MULTIPOLYGON (((22.76878 40.60366, 22.76921 40..."
2,1,2025-03-01,2025-04-01,0.084229,0.08240,0.006636,0.06660,0.1152,149,0.08024,...,0.033267,0.999509,0.444343,0.002787,0.321141,0.034030,-0.061496,0.077466,0.999509,"MULTIPOLYGON (((22.76878 40.60366, 22.76921 40..."
3,1,2025-04-01,2025-05-01,0.105649,0.09920,0.014083,0.07520,0.1354,149,0.09260,...,0.047505,0.992801,0.414077,0.003776,0.411118,0.047115,-0.025398,0.109715,0.992801,"MULTIPOLYGON (((22.76878 40.60366, 22.76921 40..."
4,1,2025-05-01,2025-06-01,0.072020,0.07070,0.007751,0.05670,0.1047,149,0.06308,...,0.053424,0.992801,0.431759,0.004336,1.006803,0.049801,0.258473,0.111319,0.986093,"MULTIPOLYGON (((22.76878 40.60366, 22.76921 40..."


## Result analysis

In [ ]:
# Reload the final table in a later session if needed.
# import pandas as pd
# results = pd.read_csv(OUTPUT_DIR / "sentinel2_monthly_parcel_statistics.csv")
# results

In [5]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if (repo_root / "src").exists():
    sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import pandas as pd

from data_analysis.eda import EDAConfig, EDAPipeline, run_eda
import webbrowser

In [9]:
tmp = results.copy()
tmp = tmp.drop(columns=["geometry"], errors="ignore")
tmp["parcel_id"] = tmp["parcel_id"].astype(str)
parcels["parcel_id"] = parcels["parcel_id"].astype(str)
df_merged = parcels.merge(tmp, on="parcel_id", how="left")
df_merged.to_file(OUTPUT_DIR / "parcels_with_statistics.gpkg", driver="GPKG")

In [ ]:
cols = list(results.columns)
cols_to_check = ["parcel_id", "period_start"] + [x for x in results.columns if x.endswith("_median")] + ["geometry"]
df = results[cols_to_check].copy()
df = df.merge(parcels[["parcel_id", "class"]], on="parcel_id", how="left", validate="many_to_one")
print(f"Number of rows in df: {len(df)}")
df.head()

In [ ]:
target_column = "class"
try:
    import geopandas as gpd
    from shapely.geometry import Point

    EDA_PATH = OUTPUT_DIR / "results_eda"

    kwargs = {
        "max_columns_for_plots": 15,
        "max_pairplot_features": 15,
        "max_multivariate_features": 15,
        "id_column": "parcel_id",
    }
    
    geo_result = run_eda(
        df,
        output_dir=str(EDA_PATH),
        target_column=target_column,
        report_title="Sample GeoDataFrame EDA",        
        **kwargs
    )
    html_path = geo_result["report_path"]
    webbrowser.open(html_path.as_uri())
except ImportError as exc:
    print(f"GeoDataFrame example skipped: {exc}")

In [ ]:
# Distribution of one feature over time, with an optional parcel overlay.
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

parcel_to_check = None #results["parcel_id"].iloc[0]  # Set to None to hide the parcel line.
date_column = "period_start"
show_outliers = False

for feature in [target_column] + [x for x in df.select_dtypes(include="number").columns if x != target_column]:
    plot_data = results[[date_column, "parcel_id", feature]].copy()
    plot_data[date_column] = pd.to_datetime(plot_data[date_column], errors="coerce")
    plot_data[feature] = pd.to_numeric(plot_data[feature], errors="coerce")
    plot_data = plot_data.dropna(subset=[date_column, feature]).sort_values(date_column)
    plot_data["period_label"] = plot_data[date_column].dt.strftime("%Y-%m-%d")

    fig_width = max(20, plot_data["period_label"].nunique() * 0.7)
    fig, ax = plt.subplots(figsize=(fig_width, 6))

    if target_column is not None:
        sns.boxplot(data=plot_data, x="period_label", y=feature, color="lightsteelblue", hue=target_column, showfliers=show_outliers, ax=ax)
        sns.stripplot(data=plot_data, x="period_label", y=feature, color="darkblue", size=3, alpha=0.5, jitter=True, ax=ax)
    else:
        sns.boxplot(data=plot_data, x="period_label", y=feature, color="lightsteelblue", showfliers=show_outliers, ax=ax)

    if parcel_to_check is not None:
        parcel_data = plot_data.loc[plot_data["parcel_id"].eq(parcel_to_check)]
        sns.lineplot(
            data=parcel_data,
            x="period_label",
            y=feature,
            color="crimson",
            marker="o",
            linewidth=2,
            label=f"Parcel {parcel_to_check}",
            sort=False,
            ax=ax,
        )

    ax.set_xlabel("Period start")
    ax.set_ylabel(feature)
    ax.set_title(f"Distribution of {feature} over time")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()
    plt.close(fig)